In [27]:
%pip install xmltodict

Note: you may need to restart the kernel to use updated packages.


In [28]:
import json
import html
import urllib
import urllib.request
import urllib.parse
import pandas as pd
import xmltodict


In [17]:
theme = "climate"
url_nn= "https://hn.algolia.com/api/v1/search?" + urllib.parse.urlencode(
    {"query": theme, "tags": "story", "hitsPerPage": 50})

print(url_nn)

with urllib.request.urlopen(url_nn) as reponse:
    d_hn = json.load(reponse)


https://hn.algolia.com/api/v1/search?query=climate&tags=story&hitsPerPage=50


In [18]:
print(d_hn.keys())
print()
print(d_hn["nbHits"])
    

dict_keys(['exhaustive', 'exhaustiveNbHits', 'exhaustiveTypo', 'hits', 'hitsPerPage', 'nbHits', 'nbPages', 'page', 'params', 'processingTimeMS', 'processingTimingsMS', 'query', 'serverTimeMS'])

18444


In [19]:
hits = d_hn["hits"]
print(len(hits))

50


In [22]:
docs = []
origines = []
for hit in hits:
    texte = hit.get("story_text")
    if not texte:
        texte = hit.get("title","")
    texte = html.unescape(texte).replace("<p>", " ").strip()
    
    
    docs.append(texte)
    origines.append("hackernews")

In [23]:
print("Documents Hacker New :", len(docs))

Documents Hacker New : 50


NameError: name 'xmltodict' is not defined

In [29]:
url_arxiv = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode(
    {"search_query": "all:" + theme, "start": 0, "max_results": 5}
)
print(url_arxiv)

http://export.arxiv.org/api/query?search_query=all%3Aclimate&start=0&max_results=5


In [30]:
with urllib.request.urlopen(url_arxiv) as reponse:
    d_arxiv = xmltodict.parse(reponse.read())

In [31]:
print(d_arxiv.keys())

dict_keys(['feed'])


In [33]:
print(d_arxiv["feed"].keys())

dict_keys(['@xmlns:opensearch', '@xmlns:arxiv', '@xmlns', 'id', 'title', 'updated', 'link', 'opensearch:itemsPerPage', 'opensearch:totalResults', 'opensearch:startIndex', 'entry'])


In [38]:
entrees = d_arxiv["feed"].get("entry", [])
type(entrees)
if isinstance(entrees, dict):
    entrees = [entrees]

print(len(entrees))

for entree in entrees:
    texte = entree["summary"].replace("\n", " ").strip()
    docs.append(texte)
    origines.append("arxiv")

print("Taille du corpus: ", len(docs))

5
Taille du corpus:  55


In [39]:
df = pd.DataFrame({
    "id": range(len(docs)),
    "texte": docs,
    "origine": origines
})


df.head()

,id,texte,origine
0,0,Climate change: IPCC report is 'code red for h...,hackernews
1,1,Climate activist arrested after ProtonMail pro...,hackernews
2,2,White House eyes subsidies for nuclear plants ...,hackernews
3,3,DOE announces breakthrough in residential cold...,hackernews
4,4,Websites hosting major US climate reports take...,hackernews


In [40]:
df.to_csv("corpus.csv", sep="\t", index=False, encoding="utf-8")
print("Fichier 'corpus.csv' sauvegardé avec succès.")

Fichier 'corpus.csv' sauvegardé avec succès.


In [41]:
df_charge = pd.read_csv("corpus.csv", sep="\t", encoding="utf-8")


print("Dimensions :", df_charge.shape)
print("\nRépartition par source :")
print(df_charge["origine"].value_counts())
df_charge.head()

Dimensions : (55, 3)

Répartition par source :
origine
hackernews    50
arxiv          5
Name: count, dtype: int64


,id,texte,origine
0,0,Climate change: IPCC report is 'code red for h...,hackernews
1,1,Climate activist arrested after ProtonMail pro...,hackernews
2,2,White House eyes subsidies for nuclear plants ...,hackernews
3,3,DOE announces breakthrough in residential cold...,hackernews
4,4,Websites hosting major US climate reports take...,hackernews


In [42]:
print(f"Nombre de documents dans le corpus : {len(df)}")

Nombre de documents dans le corpus : 55


In [43]:
for idx, row in df.iterrows():
    texte = row["texte"]
    
    # Découpage par espace
    nb_mots = len(texte.split())
    
    # Découpage par point en ignorant les résidus vides
    phrases = [p for p in texte.split(".") if p.strip()]
    nb_phrases = len(phrases)
    
    print(f"Doc {row['id']} ({row['origine']}) : {nb_mots} mots, {nb_phrases} phrase(s)")

Doc 0 (hackernews) : 9 mots, 1 phrase(s)
Doc 1 (hackernews) : 9 mots, 1 phrase(s)
Doc 2 (hackernews) : 12 mots, 1 phrase(s)
Doc 3 (hackernews) : 10 mots, 1 phrase(s)
Doc 4 (hackernews) : 8 mots, 1 phrase(s)
Doc 5 (hackernews) : 10 mots, 1 phrase(s)
Doc 6 (hackernews) : 11 mots, 2 phrase(s)
Doc 7 (hackernews) : 12 mots, 1 phrase(s)
Doc 8 (hackernews) : 5 mots, 1 phrase(s)
Doc 9 (hackernews) : 11 mots, 1 phrase(s)
Doc 10 (hackernews) : 11 mots, 1 phrase(s)
Doc 11 (hackernews) : 6 mots, 1 phrase(s)
Doc 12 (hackernews) : 11 mots, 3 phrase(s)
Doc 13 (hackernews) : 13 mots, 1 phrase(s)
Doc 14 (hackernews) : 11 mots, 1 phrase(s)
Doc 15 (hackernews) : 13 mots, 1 phrase(s)
Doc 16 (hackernews) : 4 mots, 1 phrase(s)
Doc 17 (hackernews) : 3 mots, 1 phrase(s)
Doc 18 (hackernews) : 13 mots, 1 phrase(s)
Doc 19 (hackernews) : 12 mots, 1 phrase(s)
Doc 20 (hackernews) : 11 mots, 1 phrase(s)
Doc 21 (hackernews) : 10 mots, 1 phrase(s)
Doc 22 (hackernews) : 9 mots, 3 phrase(s)
Doc 23 (hackernews) : 8 mots,

In [44]:
df = df[df["texte"].str.len() >= 20].reset_index(drop=True)

# Mise à jour des identifiants après suppression
df["id"] = range(len(df))

print(f"Nombre de documents restants : {len(df)}")

Nombre de documents restants : 54


In [45]:
corpus_texte = " ".join(df["texte"])

print(f"Longueur totale de la chaîne (en caractères) : {len(corpus_texte)}")
print(f"Aperçu (150 premiers caractères) : {corpus_texte[:150]}...")

Longueur totale de la chaîne (en caractères) : 10327
Aperçu (150 premiers caractères) : Climate change: IPCC report is 'code red for humanity' Climate activist arrested after ProtonMail provided his IP address White House eyes subsidies f...
